# 05 - Production-Grade Transforms with dbt

## What is dbt?

**dbt (data build tool)** is an open-source framework that lets data teams write SQL transformations as modular, testable, version-controlled "models". Think of it as software engineering best practices applied to data pipelines.

Instead of writing long, fragile SQL scripts that nobody understands 6 months later, dbt gives you:
- **Modularity** - each transformation is a small, focused SQL file
- **Lineage** - automatically tracks which tables depend on which (so you know what breaks if something changes)
- **Testing** - built-in data quality checks (e.g. "this column must never be NULL")
- **Documentation** - every model and column is described in plain English
- **Version control** - all code lives in Git, with full history of who changed what and when

## Why does this matter for HydraB?

In notebooks 01-03, we explored data and built Dynamic Tables interactively. That's great for prototyping, but in production you need:
- **Confidence that nothing is silently broken** - dbt tests catch issues before they reach dashboards
- **Clear ownership** - every transformation has a defined purpose and description
- **Audit trail** - when a number looks wrong, you can trace it back through the lineage graph to find where the issue is
- **Repeatable deployments** - `dbt run` rebuilds everything in the right order, every time

This is the difference between a prototype and a system the business can rely on.

## What this notebook does

1. Creates a **dbt Project** object in Snowflake from pre-staged model files
2. Executes `dbt run` to build all models (staging + marts)
3. Verifies the output tables exist and pass quality checks


## The Model Architecture

dbt models are organized in layers, each building on the previous one:

```
BRONZE (raw)          SILVER (staging)         GOLD (marts)
------------          ----------------         ------------
ASSET            -->  stg_vehicles        -->  dim_vehicle
ODOS_EVENTS      -->  stg_telemetry       -->  fct_fleet_health
DEFECT_EVENT     -->  stg_defects              
DELIVERY_TRACKING --> stg_deliveries      -->  fct_delivery_pipeline
```

**Staging models (`stg_`)** - Clean and rename raw columns. This is where we handle the messy Salesforce column names (e.g. `Chassis_Number__c` becomes `vin`) and parse the nested Odos JSON.

**Mart models (`dim_`, `fct_`)** - Business-ready tables that combine multiple staging models. These are what dashboards and analysts query directly.

| Model | Purpose | Business question it answers |
|-------|---------|-----------------------------|
| `dim_vehicle` | Vehicle dimension enriched with latest telemetry | Where is bus X right now? What's its battery level? |
| `fct_fleet_health` | Fleet health aggregated by depot | Which depot has the most vehicles with low battery? |
| `fct_delivery_pipeline` | Delivery funnel by transport type | How many buses are stuck at which port? |


## Built-in Data Quality Tests

Every model has tests defined in `schema.yml`. For example, `stg_vehicles` has:
- `vin` must be **not null** (every vehicle must have a VIN)
- `vin` must be **unique** (no duplicate vehicles)

If a test fails during `dbt run`, the pipeline stops and reports exactly which test failed and why. No more silently corrupt data reaching dashboards.


## Step 1: Set Session Context
Resolves per-user database and sets the active warehouse.


In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE WAREHOUSE HYDRAB_HOL_WH;


## Step 2: Create dbt Project

Snowflake natively supports dbt as a first-class object. Instead of running dbt from an external server (the traditional approach), we create a **dbt Project object** directly inside Snowflake.

The project files (SQL models, schema.yml, dbt_project.yml) are already staged in our internal stage from the Git repository. This command registers them as a managed dbt project.

Documentation: [dbt Projects in Snowflake](https://docs.snowflake.com/en/developer-guide/dbt/dbt-snowflake)


In [ ]:
-- Create the dbt project from pre-staged model files
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE WAREHOUSE HYDRAB_HOL_WH;

CREATE OR REPLACE DBT PROJECT GOLD.HYDRAB_FLEET_DBT
  FROM '@PUBLIC.DBT_STAGE/hydrab_fleet'
  COMMENT = 'HydraB Fleet dbt project - production transforms';


## Step 3: Execute dbt run

This is equivalent to running `dbt run` from the command line. Snowflake executes all models in dependency order:
1. First builds the staging models (since they have no upstream dependencies)
2. Then builds the mart models (which reference the staging models)

Each model materializes as a table in the appropriate schema (staging -> SILVER, marts -> GOLD).


In [ ]:
-- Execute all dbt models in dependency order
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE WAREHOUSE HYDRAB_HOL_WH;

EXECUTE DBT PROJECT GOLD.HYDRAB_FLEET_DBT
  ARGS = 'run';


## Step 4: Verify Results

Let's confirm the dbt project exists and check that our mart tables were created in the GOLD schema.


In [ ]:
-- Confirm dbt project exists
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);

SHOW DBT PROJECTS IN SCHEMA GOLD;


In [ ]:
-- Check that mart tables were built in GOLD schema
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);

SHOW TABLES IN SCHEMA GOLD;


In [ ]:
-- Preview the vehicle dimension - this is the 360 view
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);

SELECT * FROM GOLD.DIM_VEHICLE LIMIT 10;


## What You Get

With dbt deployed, HydraB now has:

- **Production-grade transforms** - every model is tested, documented, and version-controlled
- **Automatic lineage** - trace any metric back to its raw source
- **Scheduled execution** - combine with Snowflake Tasks to run on a schedule (e.g. every hour)
- **Team collaboration** - data engineers and analysts work on the same Git repo, review each other's changes

This is the foundation for a data platform that grows with the business - not one that becomes an unmaintainable mess as complexity increases.

---

**Next steps in production:**
- Schedule `EXECUTE DBT PROJECT` with a Snowflake Task (e.g. hourly refresh)
- Add more models as new data sources come online
- Run `dbt test` to validate data quality before publishing to dashboards
